# Módulo 11 de 12. DevOps y despliegue profesional

## Clase 1. Fundamentos de DevOps, Docker y contenedores

**Duración:** tres horas. **Aplicación de práctica:** backend completo de Universidad Digital, entregado por el docente **antes** de la sesión. **Resultado de la clase:** construir una imagen del backend FastAPI, iniciar API y PostgreSQL 18 con Docker Compose, comprobar persistencia y diagnosticar el sistema por sus logs.

Esta clase no incluye programar usuarios, materias, inscripciones, calificaciones ni autenticación. Esos dominios ya vienen implementados, con migración inicial, pruebas y documentación. El Módulo 9 definió sus entidades y arquitectura; esta sesión trabaja sobre el ciclo de entrega. GitHub Actions, Railway y Vercel son temas del resto del Módulo 11: hoy se establece una ejecución local reproducible que servirá de base para automatización y publicación posteriores.

Abra el cuaderno en VS Code desde la carpeta `universidad-digital` que contiene `backend/`. Las celdas Markdown contienen órdenes PowerShell y explicaciones; las dos celdas `%%writefile` permiten reconstruir el Dockerfile y Compose del laboratorio. Seleccione un kernel Python para esas celdas; todos los comandos `docker` se ejecutan en **PowerShell**, no como Python.

## 1. Aprendizajes y distribución de tiempo

Al terminar, el estudiante deberá explicar el ciclo DevOps sin reducirlo a una herramienta, distinguir una imagen de un contenedor y una VM, describir cada instrucción de un Dockerfile, verificar Docker Desktop, construir y ejecutar una imagen, levantar dos servicios con Compose, interpretar un healthcheck y recuperar datos después de reiniciar contenedores.

| Tiempo | Desarrollo de contenido y práctica | Evidencia |
|---|---|---|
| 0–20 min | Ciclo DevOps aplicado a Universidad Digital | Explicación del flujo código → prueba → imagen → operación. |
| 20–45 min | Contenedores, imágenes, VM y persistencia | Identifica qué comparte el contenedor y qué conserva el volumen. |
| 45–85 min | Instalación guiada y comprobación de Docker Desktop | `docker version`, `docker compose version`, `hello-world`. |
| 85–110 min | Dockerfile del backend explicado línea por línea | Imagen `universidad-api:clase1` construida. |
| 110–145 min | `run`, Compose, PostgreSQL y migración | API y DB saludables; Swagger disponible. |
| 145–170 min | Logs, fallas frecuentes, persistencia y asistencia IA | Diagnóstico sustentado en evidencia. |
| 170–180 min | Evaluación de salida | Entrega de comandos, URL y explicación breve. |

Si Docker Desktop ya está instalado, los 40 minutos de instalación se destinan a verificar WSL 2, reparar problemas de configuración y examinar el Dockerfile y Compose. Para una sala con internet lento, el docente puede facilitar la descarga antes de la sesión sin contar la implementación del backend dentro de estas tres horas.

## 2. ¿Qué es DevOps y por qué importa aquí?

DevOps describe prácticas de colaboración entre desarrollo y operación para entregar cambios de software con ciclos de retroalimentación cortos y controlados. La persona que desarrolla necesita saber si una nueva versión se construye, se prueba, arranca, se conecta a sus dependencias y puede observarse cuando falla. Operar el sistema aporta evidencia de vuelta a las decisiones de diseño. No existe un comando que por sí solo convierta un equipo en “DevOps”.

En Universidad Digital, el código de FastAPI y SQLAlchemy es un **artefacto fuente**. Las pruebas verifican parte de su comportamiento. El Dockerfile describe cómo construir una **imagen** ejecutable. Docker Compose describe la API, PostgreSQL, la migración y sus dependencias para el entorno local. Los logs y healthchecks permiten observar si esos procesos están funcionando. En clases posteriores, GitHub Actions automatizará pruebas y construcción; Railway y Vercel se estudiarán para publicar backend y frontend respectivamente. La entrega profesional exige gestionar secretos, migraciones y datos; una imagen correcta no sustituye estas decisiones.

Una pregunta inicial para el grupo: “Funciona en mi computador con un PostgreSQL instalado, pero falla en el equipo de otro estudiante: ¿qué elementos pueden variar?”. Respuestas posibles: versión de Python, dependencias, variables de entorno, versión de PostgreSQL, puerto, estado de migraciones y comandos de inicio. Un contenedor reduce diferencias del entorno de ejecución de la aplicación, aunque sigue dependiendo del kernel, recursos y red del anfitrión.

## 3. Contenedor, imagen y máquina virtual

Una **imagen** es el paquete construido a partir de instrucciones y capas: contiene el entorno y los archivos necesarios para iniciar la aplicación. Un **contenedor** es una instancia en ejecución de esa imagen, con su propio proceso y aislamiento. La imagen puede utilizarse varias veces para crear contenedores. El Dockerfile es una receta de construcción; no es el contenedor. `docker build` produce la imagen y `docker run` crea e inicia una instancia.

Una **máquina virtual** presenta un sistema operativo invitado completo sobre virtualización de hardware. Los contenedores Linux comparten el kernel del entorno donde corren y aíslan procesos, sistema de archivos y red; en Windows, Docker Desktop utiliza la integración de WSL 2 para ejecutar contenedores Linux. VM y contenedor no son alternativas excluyentes: los contenedores también pueden correr dentro de una VM. No afirme que todo contenedor es “más seguro” o que arranca siempre “más rápido”: dependen de configuración y carga. ([Docker: contenedores](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/); [Docker Desktop WSL 2](https://docs.docker.com/desktop/features/wsl/)).

El proceso de la API puede desaparecer al detener su contenedor. Los datos de PostgreSQL deben quedar en un **volumen** separado del sistema de archivos efímero del contenedor. En este laboratorio, el volumen `pgdata` conserva los datos cuando se usa `docker compose down` seguido de `up`. La opción `down -v` elimina también el volumen: no la utilice para la prueba de persistencia.

## 4. Instalar Docker Desktop en Windows paso a paso

Estos pasos corresponden a Windows con contenedores Linux y WSL 2. El docente debe preparar alternativas institucionales si los equipos no permiten instalar software o activar virtualización. Las pantallas pueden variar entre versiones; las comprobaciones de terminal permiten verificar el resultado. Docker mantiene instrucciones separadas para macOS y Linux en [Get Docker](https://docs.docker.com/get-started/get-docker/).

**Comprobación previa.** En Windows abra **Configuración → Sistema → Acerca de** y confirme arquitectura de 64 bits y versión de Windows compatible. Abra **Administrador de tareas → Rendimiento → CPU** y compruebe que la virtualización indique **Habilitado**. Docker documenta requisitos de WSL 2, procesador de 64 bits con SLAT, al menos 8 GB de RAM y virtualización de hardware. Si esta última está deshabilitada, solicite apoyo para activarla en BIOS/UEFI de acuerdo con la política del equipo; no cambie configuraciones institucionales sin autorización. ([Docker: instalar en Windows](https://docs.docker.com/desktop/setup/install/windows-install/)).

**WSL 2.** Abra PowerShell **como administrador** y ejecute `wsl --status`. Si WSL no está disponible, ejecute `wsl --install`, siga el mensaje de Windows y reinicie. Después ejecute `wsl --update` y vuelva a consultar `wsl --status`. Si el comando informa que la virtualización no está disponible, resuelva ese requisito antes de instalar Docker. Si la institución ya ofrece WSL 2, no repita una instalación innecesaria. ([Microsoft: instalar WSL](https://learn.microsoft.com/windows/wsl/install); [Docker: WSL](https://docs.docker.com/desktop/features/wsl/)).

**Descarga.** Entre a [Docker Desktop para Windows](https://docs.docker.com/desktop/setup/install/windows-install/) y descargue el instalador de su arquitectura (normalmente AMD64; ARM64 si el equipo usa ese procesador). Ejecute el instalador oficial. Mantenga habilitada la opción de usar el backend **WSL 2** cuando se presente. Acepte las solicitudes de Windows que correspondan a la instalación y reinicie si el asistente lo solicita. Docker Desktop incluye Engine, CLI y el plugin Compose; no necesita instalar un “docker-compose” separado para esta clase.

**Primer inicio.** Abra Docker Desktop desde Inicio, acepte los términos que se muestren y espere a que indique que el motor está en ejecución. En **Settings → General**, confirme el uso de WSL 2 si esa opción aparece. Trabajaremos con **Linux containers**. Abra una nueva ventana PowerShell normal y ejecute:

```powershell
docker version
docker compose version
docker run --rm hello-world
```

`docker version` debe mostrar cliente y servidor; si solo aparece el cliente, el motor aún no está activo. `docker compose version` comprueba el plugin. `hello-world` descarga y ejecuta una imagen pequeña; requiere acceso al registro de imágenes la primera vez. Si la red institucional bloquea esa descarga, el docente deberá facilitar una imagen aprobada. No confunda una descarga bloqueada con un Dockerfile defectuoso.

**Diagnóstico de instalación.** Si aparece “daemon is not running”, abra Docker Desktop y espere al estado listo. Si `docker` no se reconoce, reinicie PowerShell después de instalar. Si WSL requiere actualización, siga el mensaje con `wsl --update` y reinicie Docker Desktop. Si el puerto 8000 ya está ocupado, detenga la aplicación local anterior o cambie el puerto publicado en Compose.

**Si usa macOS.** Compruebe en **Acerca de este Mac** si tiene Apple silicon o Intel y que la versión de macOS esté soportada. Descargue el `Docker.dmg` correspondiente desde [Docker Desktop para Mac](https://docs.docker.com/desktop/setup/install/mac-install/). Abra el DMG, arrastre Docker a **Aplicaciones**, inicie `Docker.app`, acepte los términos y seleccione los ajustes recomendados. Espere a que el motor esté listo; en Terminal ejecute `docker version`, `docker compose version` y `docker run --rm hello-world`. Después use Terminal en lugar de PowerShell; `cp .env.example .env` reemplaza a `Copy-Item`. La guía oficial explica también los permisos de instalación.

**Si usa Ubuntu compatible.** Siga la sección **Install using the apt repository** de [Docker Engine en Ubuntu](https://docs.docker.com/engine/install/ubuntu/): compruebe la distribución soportada; elimine paquetes conflictivos enumerados allí, si los hay; actualice `apt`; cree `/etc/apt/keyrings`; importe la clave GPG oficial; añada `docker.sources` con la arquitectura y el nombre de versión de su equipo; actualice `apt`; instale `docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin`. Confirme `sudo systemctl status docker`, `sudo docker version`, `docker compose version` y `sudo docker run --rm hello-world`. La guía publica los comandos exactos y vigentes para cada paso. Para el laboratorio puede anteponer `sudo` a los comandos Docker; [Linux post-install](https://docs.docker.com/engine/install/linux-postinstall/) explica el grupo `docker`, cuyo acceso equivale a privilegios de root. Para otros Linux siga la guía oficial de su distribución. En Ubuntu `cp .env.example .env` reemplaza a `Copy-Item`.

## 5. Conocer el backend entregado antes de contenerizar

Abra `backend/README.md` y compruebe que existen `app/`, `alembic/`, `requirements.lock`, `.env.example`, `Dockerfile` y `compose.yaml`. El sistema del Módulo 9 ya implementa autenticación, roles, materias, períodos, inscripciones, calificaciones y reporte académico. La clase no los programa. El punto de entrada de la API es `app.main:app`; `app.run` permite utilizar el puerto configurado por el entorno. La migración Alembic crea el esquema inicial. Las pruebas del backend se ejecutaron antes de distribuirlo.

Para distinguir responsabilidades, responda: ¿qué archivo contiene el código de la aplicación?, ¿cuál especifica las dependencias bloqueadas?, ¿cuál describe la imagen?, ¿cuál describe varios servicios?, ¿qué archivo contiene secretos locales? El último es `.env`, que se crea desde `.env.example` y **no** se publica ni se copia a la imagen. `.dockerignore` impide enviarlo en el contexto de construcción.

El backend entregado incluye un Dockerfile de referencia para que el proyecto pueda comprobarse desde el inicio. En la siguiente celda reconstruirá ese archivo a partir de sus instrucciones y explicará cada decisión. Si ya existe, `%%writefile` lo reemplaza con el contenido mostrado; trabaje sobre una copia del paquete y conserve el original para comparar.

## 6. Primer Dockerfile para FastAPI

`FROM python:3.12-slim` elige una base Python con versión menor definida. `ENV` evita archivos de bytecode y habilita salida inmediata de logs. `WORKDIR` fija la carpeta interna. Copiar primero `requirements.lock` e instalarlo antes del código permite reutilizar la capa de dependencias cuando solo cambia una ruta Python. Se crea un usuario del sistema sin privilegios de root. Después se copian el código, la migración y la configuración. `EXPOSE` documenta el puerto interno; `CMD` inicia el proceso que escucha en `0.0.0.0`. La clase no agrega una contraseña en ninguna instrucción `RUN` ni `ENV`. ([Docker: buenas prácticas de construcción](https://docs.docker.com/build/building/best-practices/); [guía Python](https://docs.docker.com/guides/python/)).

Una base por etiqueta menor y dependencias bloqueadas mejoran la repetibilidad del ejercicio; para una cadena de suministro con exigencias más altas se puede fijar la base por digest, escanear dependencias y actualizarla de forma controlada. Una imagen pequeña no se evalúa solo por su cantidad de megabytes: también importan seguridad, compatibilidad y tiempo de construcción.

In [ ]:
%%writefile backend/Dockerfile
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

WORKDIR /app

COPY requirements.lock ./
RUN pip install --no-cache-dir --disable-pip-version-check -r requirements.lock \
    && groupadd --system appgroup \
    && useradd --system --gid appgroup --home-dir /app appuser

COPY app ./app
COPY alembic ./alembic
COPY alembic.ini ./alembic.ini

USER appuser
EXPOSE 8000
CMD ["python", "-m", "app.run"]

### Construir, inspeccionar y ejecutar una imagen

Abra PowerShell en `universidad-digital/backend` y ejecute:

```powershell
docker build -t universidad-api:clase1 .
docker image ls universidad-api
docker run --rm universidad-api:clase1 python --version
```

El punto final de `docker build` es el **contexto**: la carpeta actual que se envía a la construcción, menos las exclusiones de `.dockerignore`. La etiqueta `universidad-api:clase1` nombra la imagen local. `docker run --rm ... python --version` crea un contenedor temporal y verifica el intérprete dentro de la imagen, sin intentar aún conectar la API a PostgreSQL. Una imagen de FastAPI y una imagen de PostgreSQL tienen responsabilidades diferentes: las combinaremos con Compose.

Si el build falla al descargar la base, revise conexión y registro; si falla al instalar paquetes, lea la línea concreta del log de construcción y el `requirements.lock`. Evite introducir un token o contraseña en un prompt de IA que incluya el log completo.

## 7. Compose une API, migración y PostgreSQL

Una aplicación con base de datos necesita más de un contenedor. `compose.yaml` define `db`, `migrate` y `api`. PostgreSQL 18 guarda sus archivos en un volumen montado en `/var/lib/postgresql`, ruta de la imagen oficial para versiones 18 y posteriores. El healthcheck de `db` usa `pg_isready`; `migrate` espera a que la base esté saludable y ejecuta `alembic upgrade head`; `api` espera a que la migración termine con éxito. `depends_on` expresa orden y condiciones de arranque; no sustituye una revisión de credenciales, permisos y logs. ([Docker: Compose startup order](https://docs.docker.com/compose/how-tos/startup-order/); [imagen oficial PostgreSQL](https://hub.docker.com/_/postgres)).

La base no publica 5432 al anfitrión: la API accede a `db:5432` por la red interna de Compose. Esto evita conflictos con PostgreSQL instalado en la Clase 1 del Módulo 2. La API se publica solo en `127.0.0.1:8000` para el laboratorio local. La contraseña se toma del `.env` creado por cada estudiante y no se escribe en `compose.yaml`.

La siguiente celda reconstruye el archivo de referencia. Léalo e identifique servicio, imagen, volumen, healthcheck, dependencia y puerto.

In [ ]:
%%writefile backend/compose.yaml
services:
  db:
    image: postgres:18
    environment:
      POSTGRES_DB: ${DB_NAME:?Defina DB_NAME}
      POSTGRES_USER: ${DB_USER:?Defina DB_USER}
      POSTGRES_PASSWORD: ${DB_PASSWORD:?Defina DB_PASSWORD}
    volumes:
      - pgdata:/var/lib/postgresql
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U $$POSTGRES_USER -d $$POSTGRES_DB"]
      interval: 5s
      timeout: 3s
      retries: 15
    restart: unless-stopped

  migrate:
    build: .
    command: ["alembic", "upgrade", "head"]
    env_file: .env
    depends_on:
      db:
        condition: service_healthy
    restart: "no"

  api:
    build: .
    env_file: .env
    ports:
      - "127.0.0.1:8000:8000"
    depends_on:
      migrate:
        condition: service_completed_successfully
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8000/health/live')"]
      interval: 15s
      timeout: 5s
      retries: 5
    restart: unless-stopped

volumes:
  pgdata:

## 8. Configurar secretos locales y levantar el conjunto

En PowerShell, desde `universidad-digital/backend`, cree su archivo local y edítelo:

```powershell
Copy-Item .env.example .env
notepad .env
```

Reemplace `DB_PASSWORD`, `JWT_SECRET` (mínimo 32 caracteres) y `ADMIN_PASSWORD` (mínimo 12). Conserve `DB_HOST=db` y `DB_PORT=5432` porque la API se conectará por la red de Compose. `ADMIN_EMAIL` puede mantenerse para el laboratorio si es una dirección de ejemplo; no utilice una contraseña real de una cuenta personal. Guarde el archivo. `.env` está ignorado por Git y por el contexto Docker. No pegue su contenido en una captura ni en una consulta a IA.

Ejecute en la misma carpeta:

```powershell
docker compose config --quiet
docker compose up --build -d
docker compose ps
docker compose logs migrate
docker compose logs --tail=40 api
Invoke-RestMethod http://127.0.0.1:8000/health/ready
```

`config --quiet` valida la estructura y variables sin imprimir contraseñas. `up --build -d` construye e inicia en segundo plano. `ps` muestra estado y healthchecks; `logs` permite ver migración y arranque. El endpoint de readiness consulta PostgreSQL; una respuesta `{"database":"ready"}` demuestra conectividad. Si no está listo, consulte también `docker compose logs db` y `docker compose logs api` antes de cambiar código. ([Docker Compose](https://docs.docker.com/reference/cli/docker/compose/)).

Para inicializar roles y el primer administrador **después** de la migración:

```powershell
docker compose exec api python -m app.bootstrap
```

Abra <http://127.0.0.1:8000/docs>. En `POST /api/v1/auth/token` use el `ADMIN_EMAIL` y `ADMIN_PASSWORD` que definió. La API ya está implementada; el objetivo aquí es verificar que funciona dentro de contenedores. No envíe el token ni la contraseña al docente como evidencia. Si necesita consultar SQL directamente, use `docker compose exec db psql -U universidad -d universidad` (ajuste usuario y base si cambió esas variables).

## 9. `logs`, persistencia y diagnóstico

`docker compose logs -f api` sigue la salida de la API; detenga el seguimiento con **Ctrl+C**, que no apaga el servicio. `docker compose logs migrate` muestra si Alembic aplicó la revisión. `docker compose ps` permite comprobar estados; `docker compose exec db psql -U universidad -d universidad` abre un cliente dentro del contenedor PostgreSQL. En psql, `\dt` lista tablas y `SELECT nombre FROM roles ORDER BY nombre;` comprueba el bootstrap.

Para demostrar persistencia, cree desde Swagger una materia con el administrador. Después ejecute `docker compose down` y `docker compose up -d`. Consulte de nuevo la materia: el registro debe permanecer gracias a `pgdata`. En este laboratorio **no** use `docker compose down -v`; esa opción elimina el volumen local. Tampoco confunda `docker stop` de un contenedor con borrar una imagen o un volumen.

| Síntoma | Comprobación y acción |
|---|---|
| `docker version` muestra solo cliente | Abra Docker Desktop y espere al motor. |
| El build no descarga `python:3.12-slim` | Revise red, proxy o acceso al registro de imágenes. |
| `db` no está saludable | Consulte `docker compose logs db`; revise variables y volumen. |
| `migrate` termina con error | Consulte `docker compose logs migrate`; revise credenciales y esquema. |
| El puerto 8000 está ocupado | Detenga el proceso local o cambie el puerto del anfitrión en `ports`. |
| `/health/live` funciona y `/health/ready` falla | El proceso API corre, pero la consulta a PostgreSQL falla. |
| Swagger responde 401 o 403 | Distinga autenticación fallida de rol o asignación insuficiente. |

El uso de IA en este módulo es **asistencia supervisada**, no una fuente automática de configuración correcta. Puede solicitar una explicación de una instrucción del Dockerfile, pedir que compare dos órdenes de capas o compartir un fragmento de error **sin secretos**. Exija que la IA indique qué hipótesis se desprende del log y qué comando concreto la comprobaría. Revise cualquier propuesta en documentación oficial y ejecute de nuevo build, healthchecks y pruebas antes de aceptarla.

## 10. Actividad de asistencia IA y evaluación

**Actividad breve.** Entregue a una IA el Dockerfile de esta clase y este prompt, omitiendo `.env` y logs con valores sensibles:

> Analiza este Dockerfile de FastAPI. Identifica qué capas se invalidan si cambio solo `app/users/routes.py`, qué efecto tiene `.dockerignore`, por qué se usa `USER appuser` y propone una mejora concreta. No inventes mediciones de tamaño ni rendimiento. Separa observaciones verificables de hipótesis y cita documentación oficial.

Compare la propuesta con `docker build` y con el funcionamiento de la API. Si propone guardar contraseñas en `ARG`, `ENV` del Dockerfile, copiar `.env` a la imagen o ejecutar siempre como root, rechácela. Si recomienda fijar la imagen base por digest, explique que el digest concreto debe verificarse y mantenerse actualizado; no invente uno.

**Evidencia de salida.** El estudiante debe presentar el Dockerfile, el Compose, la salida sin secretos de `docker version`, `docker compose ps`, los estados de `/health/live` y `/health/ready`, y una consulta que demuestre persistencia tras `down` y `up`. En una explicación de 120 a 180 palabras debe diferenciar imagen, contenedor y volumen; explicar por qué `migrate` precede a `api`; y comentar una decisión de seguridad. No se califica la creación del backend: se distribuyó terminado antes de la clase.

**Marco de calidad.** La repetibilidad del build, la comprobación de salud, la protección de secretos y los logs ayudan a formular criterios de fiabilidad, seguridad y mantenibilidad del modelo ISO/IEC 25010:2023. PEP 8 corresponde al código Python entregado, mientras la especificación Compose y la documentación Docker guían la configuración de contenedores. Aplicar estos referentes no equivale a certificar el sistema. ([ISO/IEC 25010:2023](https://www.iso.org/standard/78176.html)).

### Fuentes oficiales

Docker. [Instalación Windows](https://docs.docker.com/desktop/setup/install/windows-install/), [WSL 2](https://docs.docker.com/desktop/features/wsl/), [concepto de contenedor](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/), [guía Python](https://docs.docker.com/guides/python/), [best practices](https://docs.docker.com/build/building/best-practices/) y [startup order](https://docs.docker.com/compose/how-tos/startup-order/).

PostgreSQL Official Image. [Descripción y volumen de PostgreSQL 18](https://hub.docker.com/_/postgres).

Microsoft. [Install WSL](https://learn.microsoft.com/windows/wsl/install).